In [ ]:
#!pip -q install condacolab
#import condacolab
#condacolab.install()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/gpmhc/gpmhc_train

Mounted at /content/drive
/content/drive/MyDrive/gpmhc/gpmhc_train


In [ ]:
%%bash
set -e

echo "=== Installing gpmhc environment ==="
source /usr/local/etc/profile.d/conda.sh

if conda env list | grep -q "^gpmhc "; then
    echo "Removing old gpmhc environment..."
    conda env remove -n gpmhc -y
fi

echo "Creating clean environment..."
conda create -n gpmhc python=3.10 -y
conda activate gpmhc

echo "Installing core scientific stack..."
pip install --upgrade pip
pip install \
    numpy==1.26.4 pandas==2.1.4 scipy scikit-learn matplotlib seaborn tqdm pyyaml h5py

echo "Installing PyTorch CUDA stack..."
pip install \
    torch==2.1.2+cu121 torchvision==0.16.2+cu121 torchaudio==2.1.2+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

echo "Installing graph dependencies..."
pip install dgl==1.1.3 dgllife

echo "Installing fastai stack..."
pip install fastai==2.7.14 accelerate fastcore fastdownload

echo "Installing remaining utilities..."
pip install jsonschema requests biopython

# NOTE: deliberately NOT running `pip install -e .` here - setup.py reads a
# VERSION file that doesn't exist in this repo checkout, which crashes pip's
# build-metadata step immediately (that's what "No available output" meant).
# Not needed anyway: train_HLAII_baseline_chemannot.py inserts the project
# root onto sys.path directly, so `import gpmhc` works without an editable
# install. If you want the real fix instead of skipping it:
#   echo "0.0.1" > /content/drive/MyDrive/gpmhc/gpmhc_train/VERSION
# then pip install -e . would succeed - optional, not required by anything
# in this pipeline.

echo "============================"
echo "Testing gpmhc environment"
echo "============================"
python - <<'PY'
import torch
import numpy as np
import fastai
import dgl

print("Torch:", torch.__version__, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("NumPy:", np.__version__, flush=True)
print("FastAI:", fastai.__version__, flush=True)
print("DGL:", dgl.__version__, flush=True)
print("Environment OK", flush=True)
PY

=== Installing gpmhc environment ===
Creating clean environment...
Retrieving notices: - \ done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ done

## Package Plan ##

  environment location: /usr/local/envs/gpmhc

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    icu-78.3                   |       h54a6638_2        13.8 MB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_102         728 KB  conda-forge
    libexpat-2.8.1             |       hecca717_1          76 KB  conda-forge
    libffi-3.7.0               |       h3435931_0          66 KB  conda-forge
    libgcc-16.1.



==> WARNING: A newer version of conda exists. <==
    current version: 25.11.0
    latest version: 26.7.0

Please update conda by running

    $ conda update -n base -c conda-forge conda


DGL backend not selected or invalid.  Assuming PyTorch for now.


In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc

cd /content/drive/MyDrive/gpmhc/gpmhc_train

echo "=== Files ==="
ls -la

echo
echo "=== setup.py ==="
sed -n '1,240p' setup.py

echo
echo "=== pyproject.toml ==="
cat pyproject.toml 2>/dev/null || echo "No pyproject.toml"

=== Files ===
total 1384286
-rw------- 1 root root        565 Aug  1 20:59 class_performance_comparison.csv
-rw------- 1 root root      21477 Jul 28 01:20 condacolab_install.log
drwx------ 2 root root       4096 Jul 27 21:42 dq_dataset
-rw------- 1 root root        419 Jul 27 21:01 environment.yml
drwx------ 3 root root       4096 Jul 28 00:22 experiments
drwx------ 2 root root       4096 Jul 27 19:40 gpmhc
-rw------- 1 root root       4488 Jul 28 03:03 gpmhc_environment_colab.yml
-rw------- 1 root root       3687 Jun 16 02:45 infer.py
-rw------- 1 root root          0 Jun 16 02:45 __init__.py
-rw------- 1 root root       1907 Jul 30 05:09 json_input_chemannot.json
-rw------- 1 root root       1869 Jul 24 21:16 json_input.json
drwx------ 2 root root       4096 Jul 29 04:00 metrics
-rw------- 1 root root     491008 Jun 16 02:45 mhc2_small_df.csv
-rw------- 1 root root    3786965 Jun 16 02:45 mhc_seq_df.csv
drwx------ 2 root root       4096 Jul 27 19:40 models
drwx------ 2 root root     

In [ ]:
%%bash
set -e

source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc

echo "Removing CPU DGL..."
pip uninstall -y dgl dglgo || true

echo "Installing CUDA DGL..."
pip install dgl==1.1.3+cu121 \
    -f https://data.dgl.ai/wheels/cu121/repo.html

echo "Checking DGL CUDA support..."

python - <<'PY'
import torch
import dgl

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("DGL:", dgl.__version__)

g = dgl.graph(([0,1],[1,2]))
g = g.to("cuda")
print("DGL CUDA graph transfer: OK")
PY

Removing CPU DGL...
Found existing installation: dgl 1.1.3
Uninstalling dgl-1.1.3:
  Successfully uninstalled dgl-1.1.3
Installing CUDA DGL...
Looking in links: https://data.dgl.ai/wheels/cu121/repo.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 69.2 MB/s  0:00:01
Checking DGL CUDA support...
Torch: 2.1.2+cu121
CUDA available: True
DGL: 1.1.3+cu121
DGL CUDA graph transfer: OK


In [ ]:
%%bash

nvidia-smi

source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc

python - <<'PY'
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
PY

Fri Aug  7 01:21:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             27W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc
export MPLBACKEND=Agg
echo "python: $(which python)"
python --version
cd /content/drive/MyDrive/gpmhc/gpmhc_train
python -u - <<'PY'
import sys, os
print("1. sys/os OK", flush=True)
import torch
print("2. torch OK", torch.__version__, flush=True)
import dgl
print("3. dgl OK", dgl.__version__, flush=True)
import pandas as pd
print("4. pandas OK", flush=True)

PROJECT_ROOT = os.path.abspath(".")
sys.path.insert(0, PROJECT_ROOT)
print("5. sys.path inserted:", PROJECT_ROOT, flush=True)

import gpmhc
print("6. gpmhc package OK:", gpmhc.__file__, flush=True)
from gpmhc.data import cleanup_schema, tokenize, dataset
print("7. gpmhc.data OK", flush=True)
from gpmhc.chemistry_features import get_chemistry_features
print("8. gpmhc.chemistry_features OK", flush=True)
from gpmhc.gnn_parts_chemannot import lookup_graph
print("9. gpmhc.gnn_parts_chemannot OK", flush=True)
from gpmhc.baseline_model_chemannot import model as ChemArch
print("10. gpmhc.baseline_model_chemannot OK", flush=True)
print("ALL IMPORTS OK", flush=True)
PY

python: /usr/local/envs/gpmhc/bin/python
Python 3.10.20
1. sys/os OK
2. torch OK 2.1.2+cu121
3. dgl OK 1.1.3+cu121
4. pandas OK
5. sys.path inserted: /content/drive/MyDrive/gpmhc/gpmhc_train
6. gpmhc package OK: /content/drive/MyDrive/gpmhc/gpmhc_train/gpmhc/__init__.py
7. gpmhc.data OK
8. gpmhc.chemistry_features OK
9. gpmhc.gnn_parts_chemannot OK
10. gpmhc.baseline_model_chemannot OK
ALL IMPORTS OK


<stdin>:16: DeprecationWarning: `alltrue` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `all` instead.


In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc
cd /content/drive/MyDrive/gpmhc/gpmhc_train
export MPLBACKEND=Agg
python -c "
import gpmhc.baseline_model_chemannot as chem_arch
import gpmhc.gnn_parts_chemannot as chem_parts
import gpmhc.chemistry_features as chem_feat
import gpmhc.baseline_model as base_arch  # confirm baseline still imports cleanly, untouched
print('chemistry feature order:', chem_feat.CHEMISTRY_FEATURE_NAMES)
print('chemannot pipeline module:', chem_arch.__file__)
print('baseline pipeline module (untouched):', base_arch.__file__)
"

chemistry feature order: ['hydrophobic_pair', 'polar_pair', 'positive_negative_pair', 'positive_positive_pair', 'negative_negative_pair', 'aromatic_pair']
chemannot pipeline module: /content/drive/MyDrive/gpmhc/gpmhc_train/gpmhc/baseline_model_chemannot.py
baseline pipeline module (untouched): /content/drive/MyDrive/gpmhc/gpmhc_train/gpmhc/baseline_model.py


In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc
cd /content/drive/MyDrive/gpmhc/gpmhc_train
export MPLBACKEND=Agg
python - <<'PY'
import json
from gpmhc.baseline_model import model as BaseArch
from gpmhc.baseline_model_chemannot import model as ChemArch

with open("models/baseline_model/json_input.json") as f:
    base_cfg = json.load(f)
with open("models/baseline_model/json_input_chemannot.json") as f:
    chem_cfg = json.load(f)

base_model = BaseArch(json_input=base_cfg).get_model(base_cfg["model_hyper_opts"])
chem_model = ChemArch(json_input=chem_cfg).get_model(chem_cfg["model_hyper_opts"])

print("Baseline  arch/edge_feat_size:", base_cfg["arch"], base_cfg["model_hyper_opts"]["edge_feat_size"])
print("Chemannot arch/edge_feat_size:", chem_cfg["arch"], chem_cfg["model_hyper_opts"]["edge_feat_size"])
assert chem_cfg["model_hyper_opts"]["edge_feat_size"] == base_cfg["model_hyper_opts"]["edge_feat_size"] + 6

idx = 1000
b_graph, b_edge = base_model.lookup_table[1][idx], base_model.lookup_table[2][idx]
c_graph, c_edge = chem_model.lookup_table[1][idx], chem_model.lookup_table[2][idx]
print("Baseline  graph: nodes=%d edges=%d" % (b_graph.num_nodes(), b_graph.num_edges()))
print("Chemannot graph: nodes=%d edges=%d" % (c_graph.num_nodes(), c_graph.num_edges()))
assert b_graph.num_nodes() == c_graph.num_nodes()
assert b_graph.num_edges() == c_graph.num_edges()
print("PASS: topology identical between the two independent pipelines.")
PY

Baseline  arch/edge_feat_size: baseline_model 3
Chemannot arch/edge_feat_size: baseline_model_chemannot 9
Baseline  graph: nodes=58 edges=183
Chemannot graph: nodes=58 edges=183
PASS: topology identical between the two independent pipelines.


/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(


In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate gpmhc
cd /content/drive/MyDrive/gpmhc/gpmhc_train

export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1


python -u experiments/003_HLAIIhet_chemannot/scripts/train_HLAII_baseline_chemannot.py \
  --checkpoint experiments/003_HLAIIhet_chemannot/checkpoints_epoch40/HLAII_best.pth \
  --train_csv experiments/004_NetMHC_DQaugment/dataset/HLAII_train_DQwindow_augmented.csv \
  --test_csv experiments/002_HLAII_heterodimer/dataset/HLAII_test.csv \
  --save_dir experiments/004_NetMHC_DQaugment/checkpoints \
  --metric_dir experiments/004_NetMHC_DQaugment/metrics \
  --epochs 20 \
  --lr 1e-5 \
  --batch_size 64 \
  --num_workers 2 \
  2>&1 | tee experiments/004_NetMHC_DQaugment/log.txt

Process is terminated.
